# Stage 1 PD Pipeline: Risk-Adjusted Capital Allocation

This notebook builds a production-style Probability of Default model on `accepted_loans.csv`.

Core modeling objective:

$$
\hat{p}(x) = P(\text{Default} \mid x)
$$

Class-imbalance weighting:

$$
\text{scale\_pos\_weight} = \frac{N_{\text{good}}}{N_{\text{bad}}}
$$

Calibration validation target:

$$
\hat{p}(x) \approx \Pr(\text{Default}=1 \mid \hat{p}(x)=p)
$$

The workflow includes targeted EDA, leakage prevention, missingness policy, outlier handling, Optuna tuning, SHAP explainability, and financial validation plots saved as PNG files.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import shap

from sklearn.calibration import calibration_curve
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid", context="talk")

DATA_PATH = Path("accepted_loans.csv")
PLOT_DIR = Path("plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_TRIALS = 25
MAX_ROWS_FOR_TUNING = 300_000

BAD_STATUSES = {"Charged Off", "Default", "Late (31-120 days)"}
GOOD_STATUSES = {"Fully Paid", "Current"}
TARGET_STATUSES = BAD_STATUSES.union(GOOD_STATUSES)

LEAKAGE_DROP_COLS = [
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "last_credit_pull_d",
    "last_fico_range_high",
    "last_fico_range_low",
    "hardship_flag",
    "debt_settlement_flag",
]

ADMIN_DROP_COLS = ["id", "member_id", "url", "desc", "title", "zip_code"]

c:\Users\Ashvik\projects\CLARE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def save_current_plot(path: Path) -> None:
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


def parse_int_rate(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    cleaned = series.astype("string").str.replace("%", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")


def optimize_memory(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes(include=["int64", "int32"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="float")

    object_cols = df.select_dtypes(include=["object"]).columns
    for col in object_cols:
        unique_ratio = df[col].nunique(dropna=False) / max(len(df), 1)
        if unique_ratio < 0.2:
            df[col] = df[col].astype("category")
    return df


def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=True)
    df = optimize_memory(df)

    df["loan_status"] = df["loan_status"].astype("string").str.strip()
    df["issue_year"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce").dt.year
    df["int_rate"] = parse_int_rate(df["int_rate"])
    df["dti"] = pd.to_numeric(df["dti"], errors="coerce")
    if "annual_inc" in df.columns:
        df["annual_inc"] = pd.to_numeric(df["annual_inc"], errors="coerce")
    return df


def create_target(df: pd.DataFrame) -> pd.DataFrame:
    filtered = df[df["loan_status"].isin(TARGET_STATUSES)].copy()
    filtered["default_flag"] = filtered["loan_status"].isin(BAD_STATUSES).astype("int8")
    return filtered


def plot_targeted_eda(df: pd.DataFrame) -> None:
    vintage = (
        df.dropna(subset=["issue_year"])
        .groupby("issue_year", as_index=False)["default_flag"]
        .mean()
        .rename(columns={"default_flag": "default_rate"})
    )
    vintage["default_rate"] = vintage["default_rate"] * 100

    plt.figure(figsize=(12, 6))
    sns.lineplot(data=vintage, x="issue_year", y="default_rate", marker="o", linewidth=2.5)
    plt.title("Default Rate per Year (Vintage Analysis)")
    plt.xlabel("Issue Year")
    plt.ylabel("Default Rate (%)")
    save_current_plot(PLOT_DIR / "default_rate_per_year.png")

    plt.figure(figsize=(12, 6))
    sns.histplot(
        data=df,
        x="int_rate",
        hue="loan_status",
        bins=60,
        stat="density",
        common_norm=False,
        element="step",
        alpha=0.35,
    )
    plt.title("Interest Rate Distribution by Loan Status")
    plt.xlabel("Interest Rate (%)")
    plt.ylabel("Density")
    save_current_plot(PLOT_DIR / "int_rate_distribution_by_status.png")

    plt.figure(figsize=(12, 6))
    sns.histplot(
        data=df,
        x="dti",
        hue="loan_status",
        bins=60,
        stat="density",
        common_norm=False,
        element="step",
        alpha=0.35,
    )
    plt.title("DTI Distribution by Loan Status")
    plt.xlabel("DTI")
    plt.ylabel("Density")
    save_current_plot(PLOT_DIR / "dti_distribution_by_status.png")

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [col for col in numeric_cols if col != "default_flag"]

    if numeric_cols:
        corr_target = (
            df[numeric_cols + ["default_flag"]]
            .corr(numeric_only=True)["default_flag"]
            .drop("default_flag")
            .abs()
            .sort_values(ascending=False)
        )
        top_num_features = corr_target.head(15).index.tolist()
        corr_df = df[top_num_features + ["default_flag"]].corr(numeric_only=True)

        plt.figure(figsize=(14, 10))
        sns.heatmap(corr_df, cmap="coolwarm", center=0, square=False, cbar=True)
        plt.title("Correlation Heatmap (Top Numerical Features)")
        save_current_plot(PLOT_DIR / "top_numeric_correlation_heatmap.png")

In [3]:
def apply_missingness_policy(df: pd.DataFrame, target_col: str = "default_flag"):
    missing_ratio = df.isna().mean()
    drop_cols = [c for c in missing_ratio[missing_ratio > 0.90].index if c != target_col]
    flag_cols = [c for c in missing_ratio[(missing_ratio >= 0.50) & (missing_ratio <= 0.90)].index if c != target_col]

    out = df.drop(columns=drop_cols)
    for col in flag_cols:
        if col in out.columns:
            out[f"{col}_is_missing"] = out[col].isna().astype("int8")

    return out, drop_cols, flag_cols


def clip_and_scale_outliers(df: pd.DataFrame, cols):
    used_cols = []
    for col in cols:
        if col in df.columns:
            q01, q99 = df[col].quantile([0.01, 0.99])
            if pd.notna(q01) and pd.notna(q99):
                df[col] = df[col].clip(q01, q99)
                used_cols.append(col)

    if used_cols:
        scaler = RobustScaler()
        df[used_cols] = scaler.fit_transform(df[used_cols])

    return df, used_cols


def mode_impute_categoricals(df: pd.DataFrame, categorical_cols):
    for col in categorical_cols:
        series = df[col].astype("string")
        mode_vals = series.dropna().mode()
        fill_value = mode_vals.iloc[0] if not mode_vals.empty else "MISSING"
        df[col] = series.fillna(fill_value)
    return df


def label_encode_features(df: pd.DataFrame, target_col: str = "default_flag"):
    feature_cols = [c for c in df.columns if c != target_col]
    categorical_cols = df[feature_cols].select_dtypes(include=["object", "category", "string"]).columns.tolist()

    priority = [c for c in ["grade", "sub_grade", "home_ownership"] if c in categorical_cols]
    remaining = [c for c in categorical_cols if c not in priority]
    ordered_cols = priority + remaining

    encoders = {}
    for col in ordered_cols:
        encoder = LabelEncoder()
        df[col] = encoder.fit_transform(df[col].astype("string"))
        encoders[col] = encoder

    return df, encoders

In [4]:
def fit_optuna_xgb(X_train, y_train, scale_pos_weight: float):
    if len(X_train) > MAX_ROWS_FOR_TUNING:
        X_tune, _, y_tune, _ = train_test_split(
            X_train,
            y_train,
            train_size=MAX_ROWS_FOR_TUNING,
            stratify=y_train,
            random_state=RANDOM_STATE,
        )
    else:
        X_tune, y_tune = X_train, y_train

    X_fit, X_val, y_fit, y_val = train_test_split(
        X_tune,
        y_tune,
        test_size=0.20,
        stratify=y_tune,
        random_state=RANDOM_STATE,
    )

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 900),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.30, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 10.0),
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
            "tree_method": "hist",
            "scale_pos_weight": scale_pos_weight,
        }

        model = XGBClassifier(**params)
        model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
        val_pred = model.predict_proba(X_val)[:, 1]
        return roc_auc_score(y_val, val_pred)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

    best = study.best_trial.params
    final_params = {
        "n_estimators": best["n_estimators"],
        "max_depth": best["max_depth"],
        "learning_rate": best["learning_rate"],
        "gamma": best["gamma"],
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "tree_method": "hist",
        "scale_pos_weight": scale_pos_weight,
    }

    final_model = XGBClassifier(**final_params)
    final_model.fit(X_train, y_train, verbose=False)

    return final_model, final_params, study, X_tune, y_tune


def evaluate_model(model, X_test, y_test):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.50).astype(int)
    test_auc = roc_auc_score(y_test, y_proba)

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    save_current_plot(PLOT_DIR / "confusion_matrix.png")

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.figure(figsize=(8, 6))
    sns.lineplot(x=fpr, y=tpr, linewidth=2.5, label=f"Model ROC (AUC={test_auc:.4f})")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
    plt.title("ROC-AUC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    save_current_plot(PLOT_DIR / "roc_auc_curve.png")

    prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=15, strategy="quantile")
    plt.figure(figsize=(8, 6))
    sns.lineplot(x=prob_pred, y=prob_true, marker="o", linewidth=2.0, label="Model")
    plt.plot([0, 1], [0, 1], linestyle="--", color="black", label="Perfect Calibration")
    plt.title("Calibration Curve")
    plt.xlabel("Predicted PD")
    plt.ylabel("Observed Default Frequency")
    plt.legend()
    save_current_plot(PLOT_DIR / "calibration_curve.png")

    return y_proba, test_auc


def generate_shap_outputs(model, X_test, y_proba):
    shap_sample_size = min(len(X_test), 10000)
    shap_sample = X_test.sample(n=shap_sample_size, random_state=RANDOM_STATE)

    explainer = shap.TreeExplainer(model)
    shap_explanation = explainer(shap_sample)
    shap_values = shap_explanation.values
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]

    shap.summary_plot(shap_values, shap_sample, show=False, max_display=20)
    plt.title("SHAP Summary Plot")
    save_current_plot(PLOT_DIR / "shap_summary_plot.png")

    high_risk_idx = int(np.argmax(y_proba))
    x_single = X_test.iloc[[high_risk_idx]]
    single_exp = explainer(x_single)

    single_values = single_exp.values[0]
    if np.ndim(single_values) == 2:
        single_values = single_values[:, 1]

    base_value = single_exp.base_values[0]
    if np.ndim(base_value) > 0:
        base_value = base_value[-1]

    plt.figure(figsize=(18, 4))
    shap.force_plot(base_value, single_values, x_single.iloc[0], matplotlib=True, show=False)
    plt.title("SHAP Force Plot: Highest-Risk Applicant in Test Set")
    save_current_plot(PLOT_DIR / "shap_force_plot_high_risk.png")

    return high_risk_idx

In [5]:
def run_pipeline():
    print("Loading data...")
    df = load_data(DATA_PATH)

    print("Creating target variable and filtering statuses...")
    target_df = create_target(df)
    print(f"Rows after status filtering: {len(target_df):,}")
    print(f"Default rate: {target_df['default_flag'].mean():.4%}")

    print("Generating targeted EDA plots...")
    plot_targeted_eda(target_df)

    print("Applying leakage and administrative drop rules...")
    modeling_df = target_df.copy()
    drop_cols = [c for c in LEAKAGE_DROP_COLS + ADMIN_DROP_COLS + ["loan_status"] if c in modeling_df.columns]
    modeling_df = modeling_df.drop(columns=drop_cols)

    print("Applying missingness policy (90% rule)...")
    modeling_df, dropped_90, flagged_50_90 = apply_missingness_policy(modeling_df, target_col="default_flag")

    print("Imputing missing values...")
    feature_cols = [c for c in modeling_df.columns if c != "default_flag"]
    numeric_cols = modeling_df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in feature_cols if c not in numeric_cols]

    if numeric_cols:
        num_imputer = SimpleImputer(strategy="median")
        modeling_df[numeric_cols] = num_imputer.fit_transform(modeling_df[numeric_cols])

    if categorical_cols:
        modeling_df = mode_impute_categoricals(modeling_df, categorical_cols)

    print("Clipping and robust-scaling outliers in income/debt fields...")
    modeling_df, outlier_cols_used = clip_and_scale_outliers(modeling_df, ["annual_inc", "dti"])

    print("Label-encoding categorical features (including grade/sub_grade/home_ownership)...")
    modeling_df, encoders = label_encode_features(modeling_df, target_col="default_flag")

    X = modeling_df.drop(columns=["default_flag"])
    y = modeling_df["default_flag"].astype(int)

    print("Train/test split (80/20, stratified)...")
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    n_good = int((y_train == 0).sum())
    n_bad = int((y_train == 1).sum())
    scale_pos_weight = n_good / max(n_bad, 1)
    print(f"scale_pos_weight = {scale_pos_weight:.4f}")

    print("Running Optuna hyperparameter optimization...")
    model, best_params, study, X_tune, y_tune = fit_optuna_xgb(X_train, y_train, scale_pos_weight)

    print("Evaluating model (confusion matrix, ROC-AUC, calibration)...")
    y_proba, test_auc = evaluate_model(model, X_test, y_test)

    print("Generating SHAP explainability artifacts...")
    high_risk_idx = generate_shap_outputs(model, X_test, y_proba)

    print("Computing 5-fold cross-validated AUC...")
    cv_X, cv_y = (X_tune, y_tune) if len(X_train) > MAX_ROWS_FOR_TUNING else (X_train, y_train)
    cv_model = XGBClassifier(**best_params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(cv_model, cv_X, cv_y, scoring="roc_auc", cv=cv, n_jobs=1)

    metrics = pd.DataFrame(
        {
            "metric": [
                "test_auc",
                "cv_auc_mean",
                "cv_auc_std",
                "scale_pos_weight",
                "rows_used",
                "features_used",
                "high_risk_test_index",
                "optuna_best_auc",
            ],
            "value": [
                float(test_auc),
                float(cv_scores.mean()),
                float(cv_scores.std()),
                float(scale_pos_weight),
                int(len(modeling_df)),
                int(X.shape[1]),
                int(high_risk_idx),
                float(study.best_value),
            ],
        }
    )
    metrics.to_csv(ARTIFACT_DIR / "pd_model_metrics.csv", index=False)

    pd.DataFrame(
        {
            "dropped_over_90pct_missing": pd.Series(dropped_90),
            "flagged_50_to_90pct_missing": pd.Series(flagged_50_90),
            "outlier_columns_processed": pd.Series(outlier_cols_used),
            "label_encoded_columns": pd.Series(sorted(encoders.keys())),
        }
    ).to_csv(ARTIFACT_DIR / "preprocessing_audit.csv", index=False)

    print("\nPipeline complete.")
    print(f"Test ROC-AUC: {test_auc:.4f}")
    print(f"5-fold CV AUC scores: {np.round(cv_scores, 4)}")
    print(f"5-fold CV AUC mean: {cv_scores.mean():.4f} | std: {cv_scores.std():.4f}")
    print(f"Best Optuna AUC (validation): {study.best_value:.4f}")
    print(f"Plots saved to: {PLOT_DIR.resolve()}")
    print(f"Artifacts saved to: {ARTIFACT_DIR.resolve()}")

    return {
        "model": model,
        "best_params": best_params,
        "test_auc": test_auc,
        "cv_scores": cv_scores,
        "study": study,
        "encoders": encoders,
    }

In [6]:
results = run_pipeline()

Loading data...
Creating target variable and filtering statuses...
Rows after status filtering: 2,245,134
Default rate: 12.9198%
Generating targeted EDA plots...
Applying leakage and administrative drop rules...
Applying missingness policy (90% rule)...
Imputing missing values...
Clipping and robust-scaling outliers in income/debt fields...
Label-encoding categorical features (including grade/sub_grade/home_ownership)...
Train/test split (80/20, stratified)...
scale_pos_weight = 6.7401
Running Optuna hyperparameter optimization...
Evaluating model (confusion matrix, ROC-AUC, calibration)...
Generating SHAP explainability artifacts...
Computing 5-fold cross-validated AUC...

Pipeline complete.
Test ROC-AUC: 0.7648
5-fold CV AUC scores: [0.7611 0.7563 0.7593 0.7583 0.7629]
5-fold CV AUC mean: 0.7596 | std: 0.0023
Best Optuna AUC (validation): 0.7605
Plots saved to: C:\Users\Ashvik\projects\CLARE\plots
Artifacts saved to: C:\Users\Ashvik\projects\CLARE\artifacts


<Figure size 1800x400 with 0 Axes>

In [8]:
import gc
import importlib
import time
from pathlib import Path

cp = importlib.import_module("cupy") if importlib.util.find_spec("cupy") is not None else None
torch = importlib.import_module("torch") if importlib.util.find_spec("torch") is not None else None


STABILITY_SPLIT_SEEDS = [42, 123, 2024, 7, 88]
STABILITY_MODEL_SEEDS = [42, 123, 2024, 7, 88]
STABILITY_RESULTS_PATH = Path("stability_results.csv")
RESET_STABILITY_LOG = True

PERSONA_CONFIGS = {
    "Specialist": {
        "learning_rate": 0.01,
        "n_estimators": 1000,
        "max_depth": 5,
        "gamma": 0.0,
        "subsample": 1.0,
    },
    "Generalist": {
        "learning_rate": 0.05,
        "n_estimators": 500,
        "max_depth": 4,
        "gamma": 0.0,
        "subsample": 1.0,
    },
    "Conservative": {
        "learning_rate": 0.05,
        "n_estimators": 500,
        "max_depth": 3,
        "gamma": 5.0,
        "subsample": 0.8,
    },
}


def downcast_for_rtx3050(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    float64_cols = df.select_dtypes(include=["float64"]).columns
    int64_cols = df.select_dtypes(include=["int64"]).columns

    if len(float64_cols) > 0:
        df.loc[:, float64_cols] = df[float64_cols].astype(np.float32)
    if len(int64_cols) > 0:
        df.loc[:, int64_cols] = df[int64_cols].astype(np.int32)

    return df


def clear_gpu_memory() -> None:
    gc.collect()

    if cp is not None:
        try:
            memory_pool = cp.get_default_memory_pool()
            if hasattr(memory_pool, "clear"):
                memory_pool.clear()
            else:
                memory_pool.free_all_blocks()

            pinned_pool = cp.get_default_pinned_memory_pool()
            if hasattr(pinned_pool, "free_all_blocks"):
                pinned_pool.free_all_blocks()
        except Exception:
            pass

    if torch is not None and torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass


def prepare_stage1_matrix() -> tuple[pd.DataFrame, pd.Series]:
    df = load_data(DATA_PATH)
    target_df = create_target(df)

    modeling_df = target_df.copy()
    drop_cols = [c for c in LEAKAGE_DROP_COLS + ADMIN_DROP_COLS + ["loan_status"] if c in modeling_df.columns]
    modeling_df = modeling_df.drop(columns=drop_cols)

    modeling_df, _, _ = apply_missingness_policy(modeling_df, target_col="default_flag")

    feature_cols = [c for c in modeling_df.columns if c != "default_flag"]
    numeric_cols = modeling_df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in feature_cols if c not in numeric_cols]

    if numeric_cols:
        num_imputer = SimpleImputer(strategy="median")
        modeling_df[numeric_cols] = num_imputer.fit_transform(modeling_df[numeric_cols])

    if categorical_cols:
        modeling_df = mode_impute_categoricals(modeling_df, categorical_cols)

    modeling_df, _ = clip_and_scale_outliers(modeling_df, ["annual_inc", "dti"])
    modeling_df, _ = label_encode_features(modeling_df, target_col="default_flag")

    X = modeling_df.drop(columns=["default_flag"])
    y = modeling_df["default_flag"].astype(np.int32)

    # Mandatory downcast before starting the experiment loop.
    X = downcast_for_rtx3050(X)

    return X, y


def run_stage1_gpu_stability_experiment():
    X, y = prepare_stage1_matrix()

    if RESET_STABILITY_LOG and STABILITY_RESULTS_PATH.exists():
        STABILITY_RESULTS_PATH.unlink()

    all_rows = []
    total_runs = len(STABILITY_SPLIT_SEEDS) * len(PERSONA_CONFIGS)
    run_counter = 0

    for split_seed, model_seed in zip(STABILITY_SPLIT_SEEDS, STABILITY_MODEL_SEEDS):
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.20,
            stratify=y,
            random_state=split_seed,
        )

        X_train = downcast_for_rtx3050(X_train)
        X_test = downcast_for_rtx3050(X_test)

        n_good = int((y_train == 0).sum())
        n_bad = int((y_train == 1).sum())
        scale_pos_weight = n_good / max(n_bad, 1)

        for persona_name, persona_params in PERSONA_CONFIGS.items():
            run_counter += 1
            start_time = time.perf_counter()

            try:
                model = XGBClassifier(
                    objective="binary:logistic",
                    eval_metric="auc",
                    tree_method="hist",
                    device="cuda:0",
                    max_bin=256,
                    early_stopping_rounds=50,
                    random_state=model_seed,
                    n_jobs=1,
                    scale_pos_weight=scale_pos_weight,
                    colsample_bytree=0.85,
                    **persona_params,
                )

                model.fit(
                    X_train,
                    y_train,
                    eval_set=[(X_test, y_test)],
                    verbose=False,
                )

                training_time = time.perf_counter() - start_time
                train_proba = model.predict_proba(X_train)[:, 1]
                test_proba = model.predict_proba(X_test)[:, 1]

                train_auc = roc_auc_score(y_train, train_proba)
                test_auc = roc_auc_score(y_test, test_proba)

                best_iteration = getattr(model, "best_iteration", None)
                if best_iteration is None:
                    best_iteration = int(persona_params["n_estimators"])
                else:
                    best_iteration = int(best_iteration)

                row = {
                    "split_seed": int(split_seed),
                    "model_seed": int(model_seed),
                    "persona": persona_name,
                    "train_auc": float(train_auc),
                    "test_auc": float(test_auc),
                    "auc_gap": float(train_auc - test_auc),
                    "training_time": float(training_time),
                    "best_iteration": int(best_iteration),
                }

                pd.DataFrame([row]).to_csv(
                    STABILITY_RESULTS_PATH,
                    mode="a",
                    header=not STABILITY_RESULTS_PATH.exists(),
                    index=False,
                )
                all_rows.append(row)

                print(
                    f"[{run_counter}/{total_runs}] "
                    f"split_seed={split_seed} model_seed={model_seed} "
                    f"persona={persona_name} test_auc={test_auc:.4f} "
                    f"time={training_time:.1f}s"
                )
            finally:
                clear_gpu_memory()

    results_df = pd.DataFrame(all_rows)

    stability_df = (
        results_df.groupby("persona", as_index=False)["test_auc"]
        .agg(mean_auc="mean", std_auc="std")
        .fillna({"std_auc": 0.0})
    )
    stability_df["stability_score"] = stability_df["mean_auc"] - (2.0 * stability_df["std_auc"])
    stability_df = stability_df.sort_values("stability_score", ascending=False).reset_index(drop=True)

    plt.figure(figsize=(10, 6))
    sns.boxplot(data=results_df, x="persona", y="test_auc", showfliers=False)
    sns.stripplot(data=results_df, x="persona", y="test_auc", color="black", alpha=0.45, size=4, jitter=0.15)
    plt.title("Test AUC Distribution by Persona")
    plt.xlabel("Persona")
    plt.ylabel("Test AUC")
    save_current_plot(PLOT_DIR / "stage1_persona_test_auc_boxplot.png")

    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=results_df,
        x="training_time",
        y="test_auc",
        hue="persona",
        style="persona",
        s=120,
    )
    plt.title("Training Time vs Test AUC")
    plt.xlabel("Training Time (seconds)")
    plt.ylabel("Test AUC")
    plt.legend(title="Persona")
    save_current_plot(PLOT_DIR / "stage1_training_time_vs_auc_scatter.png")

    production_winner = stability_df.iloc[0]

    print("\nStability summary:")
    print(stability_df)
    print("\nProduction Winner:")
    print(f"Persona: {production_winner['persona']}")
    print(f"Stability Score: {production_winner['stability_score']:.6f}")

    return results_df, stability_df, production_winner


stability_results_df, stability_summary_df, production_winner = run_stage1_gpu_stability_experiment()

[1/15] split_seed=42 model_seed=42 persona=Specialist test_auc=0.7616 time=46.0s
[2/15] split_seed=42 model_seed=42 persona=Generalist test_auc=0.7646 time=30.6s
[3/15] split_seed=42 model_seed=42 persona=Conservative test_auc=0.7617 time=28.8s
[4/15] split_seed=123 model_seed=123 persona=Specialist test_auc=0.7610 time=46.6s
[5/15] split_seed=123 model_seed=123 persona=Generalist test_auc=0.7637 time=30.7s
[6/15] split_seed=123 model_seed=123 persona=Conservative test_auc=0.7610 time=29.2s
[7/15] split_seed=2024 model_seed=2024 persona=Specialist test_auc=0.7614 time=46.3s
[8/15] split_seed=2024 model_seed=2024 persona=Generalist test_auc=0.7642 time=29.8s
[9/15] split_seed=2024 model_seed=2024 persona=Conservative test_auc=0.7614 time=29.2s
[10/15] split_seed=7 model_seed=7 persona=Specialist test_auc=0.7606 time=46.0s
[11/15] split_seed=7 model_seed=7 persona=Generalist test_auc=0.7634 time=29.6s
[12/15] split_seed=7 model_seed=7 persona=Conservative test_auc=0.7604 time=28.4s
[13/1